# Evaluation
## Generating ground truth
In agentic RAG, the AI assistant has at its disposal a search tool to help answer user prompts. The system makes an autonomous decision to query an external data source for additional information before prompting the LLM. It is up to the agent to determine how to invoke the search tool, and which arguments to be passed. The quality of this search mechanism has a significant impact on the system's outputs, and must be systematically evaluated.

We want the evaluation system to assess two aspects:
1. the relevance of the information retrieved from the external data source
2. the quality of the output produced by the system based on the search results

To perform this evaluation, we can collect logs from user interactions with the system and have a human review the relevance of the system outputs.

An alternative approach consists in reverse-engineering the process: for every document in the external data source, an LLM is used to generate a set of possible questions. From this, we obtain a dataset of question-document pairs (Q<sup>i</sup>, D<sup>i</sup>).
Using AI-generated query Q<sup>i</sup> as input, we assess whether the system's ensuing search includes reference document D<sup>i</sup> as part of its result set. This serves to indicate whether the search was effective.

In [1]:
from ingest import load_faq_data
faq_data = load_faq_data()

In [2]:
faq_data[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [3]:
documents = []

for doc in faq_data:
  if doc["course"] == "llm-zoomcamp":
    documents.append(doc)

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


### Single document

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
	questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
from evaluation_utils import llm_structured

In [10]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [11]:
records = []

for q in result.questions:
	records.append({
		"question": q,
		"document": doc["id"]
	})

## Generating ground truth for all docuy

In [12]:
records

[{'question': 'I found this course late — am I still allowed to join?',
  'document': '74eb249bbf'},
 {'question': 'Can I join after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I sign up now, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course and complete it normally?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do if I want a certificate after joining late?',
  'document': '74eb249bbf'}]

### All documents

In [13]:
from evaluation_utils import llm_structured_retry

In [14]:
def generate_ground_truth(doc):
	user_prompt = json.dumps(doc)

	out, usage = llm_structured_retry(
		openai_client,
		data_gen_instructions,
		user_prompt,
		Questions
	)

	results = []

	for q in out.questions:
		results.append({
			"question": q,
			"document": doc["id"]
		})

	return results, usage

In [22]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress
import os
import pandas as pd

In [ ]:
if not os.path.exists('./data/ground_truth-new.csv'):
	with ThreadPoolExecutor(max_workers=6) as pool:
		results = map_progress(pool, documents, generate_ground_truth)

	ground_truth = []
	usages = []

	for records, usage in results:
		ground_truth.extend(records)
		usages.append(usage)

	df_ground_truth = pd.DataFrame(ground_truth)
	df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)


The data file exists
